# 04 — Phase 2b: Causal Ablation (H1, H2, H3)

Tests the proposal's causal hypotheses on Gemma 3 4B IT, using the SAE features identified in Phase 1 and the SVD baseline established in Phase 2a:

- **Causal feature labeling** (professor's method): for each candidate language feature, ablate it and measure (i) MGSM accuracy delta, (ii) target-language perplexity delta. Tag as LANGUAGE / REASONING / SHARED / JUNK. Done at layer 17 on the top-5 A∩B candidates per language (~25 features).
- **H1**: top-k LANGUAGE-tagged feature ablation reproduces Zhao gains. k ∈ {1, 5, 10, 20} on a 50-prompt dev split, then best-k on full 250×5. Controls: random-feature ablation, Deng-style top-monolinguality (no causal filter).
- **H2**: SAE-targeted vs SVD trade-off — accuracy × language fidelity scatter on full 250×5.
- **H3**: layer-wise contribution at {9, 17, 22, 29} on dev split (top-10 A∩B per layer, no causal filter so the layer comparison is apples-to-apples).

**Compute budget:** ~10–12 hours on A100. Every block checkpoints so a disconnect doesn't lose progress.

## 0. Setup

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/kvrancic/nlp-project.git'
REPO_DIR = '/content/nlp-project'
if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
!pip install -q 'numpy>=2.0' langdetect -e .

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
Path('.env').write_text(f'HF_TOKEN={os.environ["HF_TOKEN"]}\n')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = Path('/content/drive/MyDrive/nlp-project-results')
    DRIVE_RESULTS.mkdir(exist_ok=True, parents=True)
except Exception:
    DRIVE_RESULTS = None

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
from itertools import product

from src.config import (
    TARGET_LANGUAGES, SAE_SUBSET_LAYERS, SAE_WIDTH_16K, RESULTS_DIR, D_MODEL,
)
from src.data import load_mgsm, parse_answer_number, compute_accuracy
from src.model import load_model_and_tokenizer, load_saes_at_layers
from src.intervention import run_generate_with_hooks, get_sae_decoder_directions
from src.evaluation import compute_perplexity

torch.manual_seed(0); np.random.seed(0)
MAX_NEW_TOKENS = 384
N_DEV = 50
N_TEST = 250
PRIMARY_LAYER = 17  # Phase 2 H3 prediction: layers 17, 22 carry most interference
K_VALUES = [1, 5, 10, 20]

## 1. Load model, SAEs, Phase 1 features, Phase 2a baseline

In [ ]:
model, tokenizer = load_model_and_tokenizer()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

saes = load_saes_at_layers(
    layers=SAE_SUBSET_LAYERS,
    width=SAE_WIDTH_16K, l0_target='medium',
)

# Phase 1
phase1 = torch.load(RESULTS_DIR / 'phase1_features.pt', weights_only=False)
intersection = phase1['intersection_features']  # {layer: {lang: [feat_indices]}}
top_A        = phase1['top_features_A']
reasoning_features = phase1['reasoning_features']
print('Phase 1 loaded.')
for layer in SAE_SUBSET_LAYERS:
    sizes = {l: len(intersection[layer][l]) for l in TARGET_LANGUAGES}
    print(f'  layer {layer}: A∩B sizes = {sizes}')

# Phase 2a (baseline + Zhao)
phase2a = torch.load(RESULTS_DIR / 'phase2_zhao_baseline.pt', weights_only=False)
baseline_results = phase2a['baseline_results']
zhao_test        = phase2a['zhao_test']
print(f'\nPhase 2a loaded. Baseline avg = {phase2a["baseline_avg"]:.3f}, '
      f'Zhao avg = {phase2a["zhao_avg"]:.3f}')

In [ ]:
# MGSM prompts (chat template) and gold
mgsm = load_mgsm(TARGET_LANGUAGES)
def make_prompt(q):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': q}], tokenize=False, add_generation_prompt=True,
    )
prompts = {l: [make_prompt(ex['question']) for ex in mgsm[l]] for l in TARGET_LANGUAGES}
golds   = {l: [ex['answer_number']        for ex in mgsm[l]] for l in TARGET_LANGUAGES}
dev_prompts = {l: prompts[l][:N_DEV] for l in TARGET_LANGUAGES}
dev_golds   = {l: golds[l][:N_DEV]   for l in TARGET_LANGUAGES}

## 2. Helpers: ablate-and-evaluate

In [ ]:
def ablation_config(layer: int, feature_indices: list[int]) -> dict:
    """Build {layer: directions} for run_generate_with_hooks."""
    if not feature_indices:
        return {}
    dirs = get_sae_decoder_directions(saes[layer], feature_indices).to(model.device)
    return {layer: dirs}

def evaluate_with_ablation(prompts_list, golds_list, layer, feature_indices,
                            positions='last', desc='ablate'):
    """Run generate with ablation hooks, parse answers, compute accuracy."""
    cfg = ablation_config(layer, feature_indices)
    if not cfg:
        # No-op: no features → just baseline-style generation.
        outputs = []
        for q in tqdm(prompts_list, desc=desc):
            inputs = tokenizer(q, return_tensors='pt').to(model.device)
            input_len = inputs['input_ids'].shape[1]
            with torch.no_grad():
                gen = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            outputs.append(tokenizer.decode(gen[0][input_len:], skip_special_tokens=True))
    else:
        outputs = run_generate_with_hooks(
            model, tokenizer, prompts_list, cfg,
            positions=positions, max_new_tokens=MAX_NEW_TOKENS,
        )
    preds = [parse_answer_number(o) for o in outputs]
    correct = [p is not None and abs(p - g) < 1e-6 for p, g in zip(preds, golds_list)]
    return {
        'accuracy': sum(correct) / max(len(correct), 1),
        'predictions': preds, 'outputs': outputs, 'correct': correct,
    }

def avg_acc(per_lang_results):
    return float(np.mean([r['accuracy'] for r in per_lang_results.values()]))

## 3. Causal feature labeling at PRIMARY_LAYER

For each top-5 A∩B candidate per language at layer 17:
- (i) MGSM accuracy delta on a 25-prompt dev slice (smaller than full dev to keep this section ≤90 min)
- (ii) Target-language perplexity delta on a 50-text held-out slice (MGSM problems 200–249 — disjoint from accuracy dev)

Then tag:
- LANGUAGE: ↑perplexity in that lang, ≈arithmetic (acc delta within ±2pp)
- REASONING: ↓arithmetic when ablated, ≈perplexity
- SHARED: hits both
- JUNK: hits neither

In [ ]:
N_LABEL_DEV = 25
TOP_PER_LANG = 5

label_dev_prompts = {l: prompts[l][:N_LABEL_DEV] for l in TARGET_LANGUAGES}
label_dev_golds   = {l: golds[l][:N_LABEL_DEV]   for l in TARGET_LANGUAGES}

# Held-out text per language: questions 200-249 (raw, not chat-templated).
ppl_texts = {l: [mgsm[l][i]['question'] for i in range(200, 250)]
             for l in TARGET_LANGUAGES}

# Baseline accuracy on label-dev (ablation-free)
label_baseline_acc = {}
for lang in TARGET_LANGUAGES:
    r = evaluate_with_ablation(
        label_dev_prompts[lang], label_dev_golds[lang], PRIMARY_LAYER, [],
        desc=f'{lang} no-ablation baseline',
    )
    label_baseline_acc[lang] = r['accuracy']
    print(f'  {lang}: baseline acc on {N_LABEL_DEV} prompts = {r["accuracy"]:.3f}')

In [ ]:
# Baseline perplexity per language
baseline_ppl = {}
for lang in TARGET_LANGUAGES:
    ppls = compute_perplexity(model, tokenizer, ppl_texts[lang])
    baseline_ppl[lang] = float(np.mean(ppls))
    print(f'  {lang}: baseline ppl = {baseline_ppl[lang]:.2f}')

In [ ]:
# For each top-5 A∩B candidate per language at PRIMARY_LAYER, run ablation
# and measure both deltas. Records → causal_labels[(lang, feat)] = {...}.
from src.model import get_decoder_layers
DECODER_LAYERS = get_decoder_layers(model)

candidates = {}  # lang -> [feat indices]
for lang in TARGET_LANGUAGES:
    inter = intersection[PRIMARY_LAYER][lang]
    if len(inter) >= TOP_PER_LANG:
        cand = inter[:TOP_PER_LANG]
    else:
        # Backfill from Method A if intersection is thin.
        extra = [f for f in top_A[PRIMARY_LAYER][lang] if f not in inter]
        cand = (inter + extra)[:TOP_PER_LANG]
    candidates[lang] = cand
    print(f'  {lang}: candidates = {cand}')

# Perplexity helper that registers an ablation hook before forward, then removes
# it. Uses get_decoder_layers so it works on Gemma 3's nested multimodal model.
def perplexity_with_ablation(layer, feat_indices, text_lang):
    cfg = ablation_config(layer, feat_indices)
    if not cfg:
        return baseline_ppl[text_lang]
    handles = []
    from src.intervention import directional_ablation
    for L, dirs in cfg.items():
        dirs_dev = dirs.to(model.device)
        def make_hook(d):
            def hook(module, input, output):
                hs = output[0]
                hs[:] = directional_ablation(hs, d)
                return (hs,) + output[1:]
            return hook
        handles.append(DECODER_LAYERS[L].register_forward_hook(make_hook(dirs_dev)))
    try:
        ppls = compute_perplexity(model, tokenizer, ppl_texts[text_lang])
    finally:
        for h in handles: h.remove()
    return float(np.mean(ppls))

causal_labels = {}
for lang in TARGET_LANGUAGES:
    for feat in candidates[lang]:
        # (i) accuracy delta on this language's label-dev split
        r = evaluate_with_ablation(
            label_dev_prompts[lang], label_dev_golds[lang],
            PRIMARY_LAYER, [feat], desc=f'{lang} f={feat}',
        )
        acc_delta = r['accuracy'] - label_baseline_acc[lang]
        # (ii) target-lang perplexity delta
        ppl_with_ablation = perplexity_with_ablation(PRIMARY_LAYER, [feat], lang)
        ppl_delta = ppl_with_ablation - baseline_ppl[lang]

        # Tagging thresholds
        ppl_threshold = 0.05 * baseline_ppl[lang]  # 5% relative bump = 'hits perplexity'
        acc_threshold = 0.04                        # 4pp absolute = 'hits arithmetic'
        hit_lang = ppl_delta > ppl_threshold
        hit_arith = acc_delta < -acc_threshold

        if hit_lang and not hit_arith:   tag = 'LANGUAGE'
        elif hit_arith and not hit_lang: tag = 'REASONING'
        elif hit_lang and hit_arith:     tag = 'SHARED'
        else:                            tag = 'JUNK'

        causal_labels[(lang, feat)] = {
            'tag': tag, 'acc_delta': acc_delta, 'ppl_delta': ppl_delta,
            'baseline_acc': label_baseline_acc[lang],
            'baseline_ppl': baseline_ppl[lang],
        }
        print(f'  {lang} f={feat}: Δacc={acc_delta:+.3f}, Δppl={ppl_delta:+.2f} → {tag}')
        torch.save(causal_labels, RESULTS_DIR / 'phase2b_causal_labels_partial.pt')

# Confirmed LANGUAGE features per lang (used for H1)
confirmed_language = {
    lang: [feat for (l, feat), v in causal_labels.items()
           if l == lang and v['tag'] == 'LANGUAGE']
    for lang in TARGET_LANGUAGES
}
for lang in TARGET_LANGUAGES:
    print(f'  {lang}: confirmed LANGUAGE features = {confirmed_language[lang]}')

## 4. H1 — Top-k LANGUAGE-feature ablation reproduces Zhao gains?

Sweep k on the dev split, then evaluate best k on full 250×5. Top-k preference order: confirmed LANGUAGE (causal) → A∩B intersection → Method A only.

In [ ]:
def select_top_features(lang, k, layer=PRIMARY_LAYER):
    """Prefer causally-confirmed LANGUAGE features, then A∩B, then top-A."""
    out = list(confirmed_language[lang])
    for f in intersection[layer][lang]:
        if f not in out: out.append(f)
    for f in top_A[layer][lang]:
        if f not in out: out.append(f)
    return out[:k]

h1_sweep = {}  # k -> {lang: result}
for k in K_VALUES:
    h1_sweep[k] = {}
    for lang in TARGET_LANGUAGES:
        feats = select_top_features(lang, k)
        r = evaluate_with_ablation(
            dev_prompts[lang], dev_golds[lang], PRIMARY_LAYER, feats,
            desc=f'H1 k={k} {lang}',
        )
        h1_sweep[k][lang] = r
        print(f'  k={k:>2} {lang}: {r["accuracy"]:.3f}')
    h1_sweep[k]['avg'] = avg_acc({l: h1_sweep[k][l] for l in TARGET_LANGUAGES})
    print(f'  k={k:>2} AVG: {h1_sweep[k]["avg"]:.3f}')
    torch.save(h1_sweep, RESULTS_DIR / 'phase2b_h1_sweep_partial.pt')

best_k = max(K_VALUES, key=lambda k: h1_sweep[k]['avg'])
print(f'\nBest k on dev: {best_k} (avg {h1_sweep[best_k]["avg"]:.3f})')

## 5. H1 controls — random + Deng-style at best_k, dev split

In [ ]:
rng = np.random.default_rng(0)
n_features = saes[PRIMARY_LAYER].cfg.d_sae

# Random control: ablate best_k random features, fixed seed for reproducibility.
random_features = rng.choice(n_features, size=best_k, replace=False).tolist()
ctrl_random = {}
for lang in TARGET_LANGUAGES:
    r = evaluate_with_ablation(
        dev_prompts[lang], dev_golds[lang], PRIMARY_LAYER, random_features,
        desc=f'H1-rand {lang}',
    )
    ctrl_random[lang] = r
    print(f'  random k={best_k} {lang}: {r["accuracy"]:.3f}')

# Deng-style control: ablate top-best_k Method A features (no causal filter,
# matches Deng et al.'s pure monolinguality criterion).
ctrl_deng = {}
for lang in TARGET_LANGUAGES:
    feats = top_A[PRIMARY_LAYER][lang][:best_k]
    r = evaluate_with_ablation(
        dev_prompts[lang], dev_golds[lang], PRIMARY_LAYER, feats,
        desc=f'H1-deng {lang}',
    )
    ctrl_deng[lang] = r
    print(f'  deng-style k={best_k} {lang}: {r["accuracy"]:.3f}')

## 6. H1/H2 final — best SAE config on full 250×5

In [ ]:
h1_test = {}
for lang in TARGET_LANGUAGES:
    feats = select_top_features(lang, best_k)
    print(f'\n=== H1 final: {lang} (k={best_k}, layer={PRIMARY_LAYER}) ===')
    r = evaluate_with_ablation(
        prompts[lang], golds[lang], PRIMARY_LAYER, feats,
        desc=f'H1-final {lang}',
    )
    h1_test[lang] = r
    print(f'  baseline {baseline_results[lang]["accuracy"]:.3f} → '
          f'sae-ablate {r["accuracy"]:.3f}  '
          f'(zhao {zhao_test[lang]["accuracy"]:.3f})')
    torch.save(h1_test, RESULTS_DIR / 'phase2b_h1_test_partial.pt')
h1_test_avg = avg_acc(h1_test)
print(f'\nH1-final avg: {h1_test_avg:.3f}')

## 7. H2 — language fidelity (LaBSE-free; reuse langdetect)

In [ ]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
LANG_OK = {'en': {'en'}, 'zh': {'zh-cn', 'zh-tw'}, 'es': {'es'}, 'bn': {'bn'}, 'sw': {'sw'}}

def language_fidelity(outputs, target_lang):
    ok = total = 0
    for o in outputs:
        s = (o or '')[:300].strip()
        if not s: continue
        try: d = detect(s)
        except Exception: continue
        total += 1
        if d in LANG_OK[target_lang]: ok += 1
    return ok / total if total else 0.0

fidelity = {}
for lang in TARGET_LANGUAGES:
    fidelity[lang] = {
        'baseline': language_fidelity(baseline_results[lang]['outputs'], lang),
        'zhao':     language_fidelity(zhao_test[lang]['outputs'],     lang),
        'sae':      language_fidelity(h1_test[lang]['outputs'],       lang),
    }
    print(f'  {lang}: baseline={fidelity[lang]["baseline"]:.3f}, '
          f'zhao={fidelity[lang]["zhao"]:.3f}, sae={fidelity[lang]["sae"]:.3f}')

## 8. H3 — Layer-wise contribution

Same selection method (top-10 A∩B → top-A backfill) at each of {9, 17, 22, 29} on dev split. We expect layers 17 and 22 to drive the most accuracy delta if the three-phase model holds for Gemma 3.

In [ ]:
K_H3 = 10

def select_layer_features(layer, lang, k):
    inter = intersection[layer][lang]
    extras = [f for f in top_A[layer][lang] if f not in inter]
    return (inter + extras)[:k]

h3_results = {}  # layer -> {lang: result}
for layer in SAE_SUBSET_LAYERS:
    h3_results[layer] = {}
    for lang in TARGET_LANGUAGES:
        feats = select_layer_features(layer, lang, K_H3)
        r = evaluate_with_ablation(
            dev_prompts[lang], dev_golds[lang], layer, feats,
            desc=f'H3 L{layer} {lang}',
        )
        h3_results[layer][lang] = r
        print(f'  layer {layer} {lang}: {r["accuracy"]:.3f}')
    h3_results[layer]['avg'] = avg_acc({l: h3_results[layer][l] for l in TARGET_LANGUAGES})
    print(f'  layer {layer} AVG: {h3_results[layer]["avg"]:.3f}')
    torch.save(h3_results, RESULTS_DIR / 'phase2b_h3_partial.pt')

## 9. Figures

In [ ]:
FIG_DIR = Path('results/figures'); FIG_DIR.mkdir(exist_ok=True, parents=True)
sns.set_theme(style='whitegrid', context='paper')

# Figure 1: H1 sweep — accuracy vs k per language.
fig, ax = plt.subplots(figsize=(7, 3.5))
for lang in TARGET_LANGUAGES:
    ys = [h1_sweep[k][lang]['accuracy'] for k in K_VALUES]
    ax.plot(K_VALUES, ys, marker='o', label=lang)
ax.axhline(y=phase2a['baseline_avg'], ls='--', color='grey', label='baseline avg')
ax.set_xlabel('k (number of LANGUAGE features ablated)')
ax.set_ylabel('MGSM accuracy')
ax.set_title(f'H1: accuracy vs k at layer {PRIMARY_LAYER} (dev split)')
ax.legend(loc='lower right')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig7_h1_sweep.png', dpi=150); plt.show()

In [ ]:
# Figure 2: H1/H2 final comparison (baseline / zhao / sae) per language.
fig, ax = plt.subplots(figsize=(8, 3.5))
df = pd.DataFrame({
    'baseline': {l: baseline_results[l]['accuracy'] for l in TARGET_LANGUAGES},
    'zhao SVD': {l: zhao_test[l]['accuracy'] for l in TARGET_LANGUAGES},
    f'sae k={best_k}': {l: h1_test[l]['accuracy'] for l in TARGET_LANGUAGES},
})
df.plot(kind='bar', ax=ax)
ax.set_ylabel('MGSM accuracy'); ax.set_ylim(0, 1)
ax.set_title('H1/H2: SAE-targeted ablation vs Zhao SVD vs unmodified (full 250×5)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig8_h1h2_final.png', dpi=150); plt.show()

In [ ]:
# Figure 3: H2 — accuracy vs language fidelity scatter
fig, ax = plt.subplots(figsize=(5.5, 4))
for lang, marker in zip(TARGET_LANGUAGES, ['o', 's', '^', 'D', 'v']):
    ax.scatter(fidelity[lang]['baseline'], baseline_results[lang]['accuracy'],
               marker=marker, s=70, color='grey',  label=f'{lang} baseline')
    ax.scatter(fidelity[lang]['zhao'],     zhao_test[lang]['accuracy'],
               marker=marker, s=70, color='C0',    label=f'{lang} zhao')
    ax.scatter(fidelity[lang]['sae'],      h1_test[lang]['accuracy'],
               marker=marker, s=70, color='C3',    label=f'{lang} sae')
ax.set_xlabel('language fidelity (output langdetect == input lang)')
ax.set_ylabel('MGSM accuracy')
ax.set_title('H2: accuracy × fidelity trade-off')
ax.legend(fontsize=7, loc='lower left', ncol=3)
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig9_h2_tradeoff.png', dpi=150); plt.show()

In [ ]:
# Figure 4: H3 layer-wise accuracy.
fig, ax = plt.subplots(figsize=(7, 3.5))
df = pd.DataFrame({
    f'L{layer}': {l: h3_results[layer][l]['accuracy'] for l in TARGET_LANGUAGES}
    for layer in SAE_SUBSET_LAYERS
})
df.T.plot(kind='bar', ax=ax)
ax.set_ylabel('MGSM accuracy'); ax.set_ylim(0, 1)
ax.set_title(f'H3: top-{K_H3} ablation per layer (dev split)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig10_h3_layers.png', dpi=150); plt.show()

## 10. Bootstrap CIs and save

In [ ]:
def bootstrap_ci(correct_list, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    arr = np.asarray(correct_list, dtype=float)
    boots = [arr[rng.integers(0, len(arr), size=len(arr))].mean() for _ in range(n_boot)]
    return (float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)))

ci_summary = {}
for lang in TARGET_LANGUAGES:
    ci_summary[lang] = {
        'baseline': bootstrap_ci(baseline_results[lang]['correct']),
        'zhao':     bootstrap_ci(zhao_test[lang]['correct']),
        'sae':      bootstrap_ci(h1_test[lang]['correct']),
    }
    base_mean = baseline_results[lang]['accuracy']
    z_mean    = zhao_test[lang]['accuracy']
    s_mean    = h1_test[lang]['accuracy']
    print(f'{lang}: baseline {base_mean:.3f} {ci_summary[lang]["baseline"]} | '
          f'zhao {z_mean:.3f} {ci_summary[lang]["zhao"]} | '
          f'sae {s_mean:.3f} {ci_summary[lang]["sae"]}')

In [ ]:
phase2b_payload = {
    'config': {
        'primary_layer': PRIMARY_LAYER, 'k_values': K_VALUES, 'best_k': best_k,
        'k_h3': K_H3, 'n_dev': N_DEV, 'n_test': N_TEST,
        'max_new_tokens': MAX_NEW_TOKENS,
        'random_features_seed_0': random_features,
    },
    'causal_labels': causal_labels,
    'confirmed_language': confirmed_language,
    'baseline_acc_label_dev': label_baseline_acc,
    'baseline_ppl': baseline_ppl,
    'h1_sweep': h1_sweep,
    'ctrl_random': ctrl_random,
    'ctrl_deng': ctrl_deng,
    'h1_test': h1_test,
    'fidelity': fidelity,
    'h3_results': h3_results,
    'ci_summary': ci_summary,
    'best_k': best_k,
    'h1_test_avg': h1_test_avg,
}
out = RESULTS_DIR / 'phase2_ablation.pt'
torch.save(phase2b_payload, out)
print(f'Saved {out} ({out.stat().st_size/1e6:.1f} MB)')
if DRIVE_RESULTS is not None:
    torch.save(phase2b_payload, DRIVE_RESULTS / 'phase2_ablation.pt')

print('\n=== Phase 2b summary ===')
print(f'Causal labels by tag:')
from collections import Counter
tag_counts = Counter(v['tag'] for v in causal_labels.values())
print(f'  {dict(tag_counts)}')
print(f'Best k = {best_k}')
print(f'Avg accuracy:')
print(f'  baseline: {phase2a["baseline_avg"]:.3f}')
print(f'  zhao:     {phase2a["zhao_avg"]:.3f}')
print(f'  sae k={best_k}: {h1_test_avg:.3f}')
print(f'  random k={best_k} (dev): {avg_acc(ctrl_random):.3f}')
print(f'  deng-style k={best_k} (dev): {avg_acc(ctrl_deng):.3f}')